# 101 heatmap puzzles

A companion set to [100 pandas puzzles](https://github.com/ajtruijas/100-pandas-puzzles), covering `seaborn` heatmaps and the pandas reshaping that feeds them.

Unlike the earlier puzzles, everything here starts from *dirty* data. That is deliberate: a heatmap is only as trustworthy as the matrix underneath it, so the exercises are ordered **ingest -> clean -> engineer -> reshape -> plot**, and each heatmap question sits after the cleaning question that makes it answerable.

Difficulty ratings are the same as before: *easy* -> *medium* -> *hard*.

## The dataset

Twenty music bands formed between 1960 and 2021, spanning seven decades. The numeric columns are **illustrative figures for exercising the techniques**, not authoritative music-industry statistics -- `peak_streams` in particular is a made-up scale. The *structure* of the data is what matters here, and the structure is deliberately awkward:mixed-case and padded strings, a comma-and-suffix-encoded number stored as text, and a column of missing values. Each of those maps onto a real ingestion bug you will meet again immediately outside this notebook.

Boilerplate for all plotting cells:
```python
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('ggplot')
```

## Tidy the data

### Making a DataFrame safe to plot

Difficulty: *easy* -> *medium*

Every heatmap in the second half of this notebook fails if this section is skipped or done badly. Work through it in order.

**61.** Build a DataFrame `df` from the dictionary below. Do not clean anything yet -- we want to see the raw state.

In [ ]:
df = pd.DataFrame({
    'band': ['The Beatles', 'The Rolling Stones', 'Pink Floyd', 'Led Zeppelin',
             'Queen', 'ABBA', 'U2', 'The Cure', 'Depeche Mode', 'Metallica',
             'a-ha', 'Oasis', 'Radiohead', 'Foo Fighters', 'Arctic Monkeys',
             'Tame Impala', 'Glass Animals', 'Blackpink', 'Wet Leg',
             'Chappell Roan'],
    'formed': [1960, 1962, 1965, 1968, 1970, 1972, 1976, 1978, 1980, 1981,
               1982, 1991, 1991, 1994, 2002, 2007, 2010, 2016, 2021, 2020],
    'country': [' england ', 'England', 'england', 'England', 'England',
                'Sweden', 'Ireland', 'England', 'England', 'USA',
                'Norway', 'England', 'England', 'usa', 'England',
                'Australia', 'England', 'South Korea', 'England', 'USA'],
    'genre': ['Rock', 'rock', 'Progressive Rock', 'Hard Rock', 'Glam Rock',
              'Pop', 'Rock', 'Alternative', 'Synth-pop', 'Thrash Metal',
              'Pop Rock', 'Britpop', 'Alternative', 'Alternative',
              'Indie Rock', 'Psychedelic', 'Indie Pop', 'K-pop',
              'Indie Rock', 'Pop'],
    'studio_albums': [12, 30, 15, 9, 15, 11, 14, 15, 15, 11, 9, 10, 10, 11,
                      7, 5, 4, 2, 2, 2],
    'best_chart_position': [1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1,
                            4, 1, 2, 1],
    'peak_streams': ['780.5M', '1,240.3M', '980.2M', '890.6M', '1450.9M',
                     '860.4M', '1120.7M', '610.8M', '1240.5M', '1380.2M',
                     '540.3M', '980.6M', '1320.4M', '940.7M', '890.1M',
                     '920.3M', '760.5M', '1150.9M', '480.2M', '1560.4M'],
    'grammy_awards': [7, 3, 1, 0, 4, 1, 22, 3, 2, 9, 0, 4, 3, 6, 4, 1, 0, 0,
                      0, 2],
    'years_since_peak': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan,
                         9, 4, 12, 17, 6, 3, 11, 8, 2, np.nan, np.nan,
                         np.nan, np.nan, np.nan],
})
df


**62.** Inspect the result. Print the dtypes and the first few rows, then answer these in a comment or by running the checks:

- Which columns would break a numeric heatmap if passed straight to `sns.heatmap()`?
- How many distinct values does `country` appear to have? Does that match reality?
- How much of `years_since_peak` is missing?

In [ ]:
print(df.dtypes)
df.head(8)


In [ ]:
# distinct country values -- more than there really are
print(df['country'].nunique(), 'distinct country values')
print(sorted(df['country'].unique()))
print('missing share of years_since_peak: {:.0%}'.format(df['years_since_peak'].isna().mean()))


**63.** Clean the `country` and `genre` columns: strip surrounding whitespace and normalise capitalisation so that every band from England is `England`, not `england` or `' england '`.

Do it with a single vectorised string operation per column, not a loop. *Hint: `Series.str.strip()` and `Series.str.title()`.*

In [ ]:
df['country'] = df['country'].str.strip().str.title()
df['genre'] = df['genre'].str.strip().str.title()
print(df['country'].value_counts())
print(df['genre'].value_counts())


**64.** `peak_streams` is text like `'780.5M'` and `'1,240.3M'`. Convert it to a real number in millions, stored in a new column called `peak_streams_m`.

*Hint: `Series.str.replace` with a character class `[M,]`, then `astype(float)`.* Leave the original column alone.

In [ ]:
df['peak_streams_m'] = (df['peak_streams']
                        .str.replace(r'[M,]', '', regex=True)
                        .astype(float))
df[['peak_streams', 'peak_streams_m']].head()


**65.** Missing data in `years_since_peak` is *not* random -- it is concentrated in the classic bands, whose peaks predate the streaming era being measured.

Measure the damage before you act, then impute rather than drop. Report the proportion missing, then fill it with the median of the bands that *do* have a value.

In [ ]:
missing = df['years_since_peak'].isna()
print('missing: {} of {} rows ({:.0%})'.format(missing.sum(), len(df), missing.mean()))
print(df.loc[missing, ['band', 'formed']].to_string(index=False))


In [ ]:
median_ysp = df['years_since_peak'].median()
df['years_since_peak'] = df['years_since_peak'].fillna(median_ysp)
print('median used: {}'.format(median_ysp))
print(df['years_since_peak'].isna().sum(), 'values still missing')


**66.** Derive three features that the heatmaps will need:

1. `decade` -- the formation decade as a label string, e.g. `'1970s'`.
2. `years_active` -- `2025 - formed`.
3. `chart_score` -- `11 - best_chart_position`, so that a *higher* value means a *better* chart position. (Chart positions are rank-1-best; most metrics grow upward. Inverting the sign is what lets one colormap read consistently across every panel.)

*Hint: integer floor division, `//`, then `astype(str) + 's'`.*

In [ ]:
df['decade'] = ((df['formed'] // 10) * 10).astype(str) + 's'
df['years_active'] = 2025 - df['formed']
df['chart_score'] = 11 - df['best_chart_position']
df[['band', 'formed', 'decade', 'years_active', 'best_chart_position', 'chart_score']].head()


**67.** Aggregate by decade. For each decade, report the number of bands, the mean peak streams, and the total Grammy wins, ordered chronologically.

In [ ]:
by_decade = (df.groupby('decade')
              .agg(bands=('band', 'size'),
                   mean_streams_m=('peak_streams_m', 'mean'),
                   total_grammys=('grammy_awards', 'sum'),
                   mean_years_active=('years_active', 'mean'))
              .round(1))
by_decade


## Heatmaps

### Reshaping data into a matrix worth plotting

Difficulty: *medium* -> *hard*

A heatmap needs a **rectangular, fully numeric matrix**: one factor on each axis and a single measure filling the cells. Most of the work is in producing it, not in calling `sns.heatmap()`. Work through 68 to 73 in order; each builds on the last.

**68.** A **pivot heatmap** answers 'which country dominates which decade'. Build a matrix of mean `peak_streams_m` with decades down the rows and countries across the columns.

Most cells will be `NaN` -- that is expected and it is what makes the remaining cells readable. Display it as text first; we plot it in 69.

In [ ]:
country_decade = df.pivot_table(index='decade',
                                  columns='country',
                                  values='peak_streams_m',
                                  aggfunc='mean')
country_decade.round(0)


**69.** Plot the matrix from 68 as a heatmap with `sns.heatmap`.

Requirements: decades ordered oldest to top; readable labels; the empty cells left blank rather than drawn as zero; and a colourbar whose label actually says what the numbers mean.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(country_decade,
            annot=True, fmt='.0f',
            cmap='YlGnBu',
            linewidths=0.5,
            mask=country_decade.isna(),
            cbar_kws={'label': 'Mean peak streams (millions)'},
            ax=ax)
ax.set_title('Mean peak streams by country and formation decade')
ax.set_xlabel('Country')
ax.set_ylabel('Formation decade')
plt.tight_layout()
plt.show()


**70.** The countries in 69 are ordered alphabetically, so the eye has to hunt. Reorder them by total streams and re-plot.

On a single categorical axis the ordering is a display choice, so pick it deliberately: `reindex` the columns by their column totals, descending. The matrix's cells do not move; only their labels do.

In [ ]:
ordered_cols = country_decade.sum().sort_values(ascending=False).index
country_decade_ordered = country_decade[ordered_cols]

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(country_decade_ordered,
            annot=True, fmt='.0f',
            cmap='YlGnBu',
            linewidths=0.5,
            mask=country_decade_ordered.isna(),
            cbar_kws={'label': 'Mean peak streams (millions)'},
            ax=ax)
ax.set_title('Mean peak streams by country, heaviest first')
ax.set_xlabel('Country')
ax.set_ylabel('Formation decade')
plt.tight_layout()
plt.show()


**71.** A **feature correlation heatmap** is the other standard form: every row and column is the same set of variables, so the matrix is square and symmetric.

Build the correlation matrix across the numeric columns only, then plot it with a **diverging** colormap centred on zero. A sequential map would be wrong here: zero correlation is a meaningful midpoint, not 'a small amount of something'.

*Hint: `corr()`, `cmap='RdBu_r'`, and `center=0`.*

In [ ]:
df['decade_code'] = df['formed'] // 10 * 10

features = ['formed', 'decade_code', 'years_active', 'studio_albums',
            'best_chart_position', 'chart_score', 'peak_streams_m',
            'grammy_awards', 'years_since_peak']
corr = df[features].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr,
            annot=True, fmt='.2f',
            cmap='RdBu_r',
            center=0,
            vmin=-1, vmax=1,
            square=True,
            linewidths=0.5,
            cbar_kws={'label': "Pearson's r", 'shrink': 0.8},
            ax=ax)
ax.set_title('Feature correlations across 20 bands')
plt.tight_layout()
plt.show()


**72.** A square correlation matrix plots its data twice, mirrored about the diagonal. Hide the duplicate with `mask` and `np.triu`, then note which pair dominates the plot.

Masking is not cosmetic here: it is what makes a large matrix legible at all.

In [ ]:
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr,
            mask=mask,
            cmap='RdBu_r',
            center=0,
            vmin=-1, vmax=1,
            square=True,
            linewidths=0.5,
            cbar_kws={'label': "Pearson's r", 'shrink': 0.8},
            ax=ax)
ax.set_title('Feature correlations (lower triangle only)')
plt.tight_layout()
plt.show()


**73.** Raw values across features with different units (years vs. millions vs. counts) cannot share a colour scale. Standardise each column to a 0-1 range, then plot the **decade x feature** mean matrix.

Min-max scaling is the minimum viable fix; for skewed measures a rank or z-score transform would be better.

*Hint: `DataFrame.min(axis=0)` / `max(axis=0)`, or `sklearn.preprocessing.MinMaxScaler`.* Watch out for division by zero on constant columns.

In [ ]:
scaled = df[features].copy()
denom = scaled.max() - scaled.min()
denom[denom == 0] = 1            # guard against constant columns
scaled = (scaled - scaled.min()) / denom

decade_features = scaled.assign(decade=df['decade']).groupby('decade').mean()
decade_features


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(decade_features,
            annot=True, fmt='.2f',
            cmap='mako',
            linewidths=0.5,
            cbar_kws={'label': 'Scaled value (0-1 within each feature)'},
            ax=ax)
ax.set_title('Average scaled profile of each decade')
ax.set_xlabel('Feature')
ax.set_ylabel('Formation decade')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


**74.** Finally, let the data choose the row and column order instead of you. `sns.clustermap` hierarchically clusters both axes so similar decades and similar features end up adjacent.

Read the dendrogram rather than the cells: features that merge early are measuring nearly the same thing. `best_chart_position` and `chart_score` should merge almost immediately, and their correlation matrix entry should confirm it.

In [ ]:
g = sns.clustermap(scaled,
                 cmap='mako',
                 standard_scale=1,
                 figsize=(10, 8),
                 linewidths=0.4,
                 cbar_pos=(0.02, 0.8, 0.05, 0.18),
                 annot=False)
g.fig.suptitle('Clustered features, standardised to 0-1', y=1.02)
plt.show()


In [ ]:
print('chart_score vs best_chart_position: r = {:.3f}'.format(
    corr.loc['chart_score', 'best_chart_position']))
print('formed vs years_active:          r = {:.3f}'.format(
    corr.loc['formed', 'years_active']))


## Where to go next

- Swap `aggfunc='mean'` for `'median'`, `'count'` or `'sum'` in 68 and watch which cells change.
- Try `pd.crosstab` and `pd.melt` for 68; they solve the same reshape differently and you will want both.
- In 71, drop `formed` and `decade_code` -- they are the same information twice, and a correlation heatmap will happily plot that redundancy as a strong 'relationship'.
- For real missing data, `sns.heatmap` on a matrix with gaps is misleading; consider `mask` plus an explicit annotation of how many observations each cell averages.
- `sns.heatmap` will happily accept a non-numeric matrix and produce a picture rather than an error. If a panel comes out uniformly grey, suspect `object` dtype before you suspect the colormap.